In [ ]:
import pandas as pd
import re
import requests
from datetime import datetime
from collections import defaultdict
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
def get_multilingual_urls(en_urls):
    all_data = []

    headers = {'User-Agent': 'Mozilla/5.0'}
    api_url = "https://en.wikipedia.org/w/api.php"

    for en_url in en_urls:
        title = en_url.split("/wiki/")[-1]
        params = {
            "action": "query",
            "prop": "langlinks",
            "titles": title,
            "format": "json",
            "lllimit": "500"
        }

        response = requests.get(api_url, params=params, headers=headers)
        if response.status_code != 200:
            continue  # O puedes guardar un error aquí

        data = response.json()
        pages = data.get("query", {}).get("pages", {})

        for page in pages.values():
            langlinks = page.get("langlinks", [])

            for link in langlinks:
                lang_code = link["lang"]
                lang_title = link["*"].replace(" ", "_")
                multilingual_url = f"https://{lang_code}.wikipedia.org/wiki/{lang_title}"

                all_data.append({
                    "language": lang_code,
                    "multilingual_url": multilingual_url,
                    "source_url": en_url
                })

            # Añadir versión en inglés también
            all_data.append({
                "language": "en",
                "multilingual_url": en_url,
                "source_url": en_url
            })

    return pd.DataFrame(all_data)


def wikipedia_views(wikipedia_urls):
    
    df_views = pd.DataFrame({'page_title': wikipedia_urls})
    
    for count, url in enumerate(wikipedia_urls, 1):
        print(f"{round(100 * count / len(wikipedia_urls), 2)} % procesado", end='\r')
        
        try:
            lang_code = re.sub(r'\..*', '', url.split('//')[1])  # ej: "en" de "en.wikipedia.org"
            article = re.sub(r'^.*?org/wiki/', '', url)          # ej: "Cat"
            api_url = (
                f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
                f"{lang_code}.wikipedia/all-access/user/"
                f"{article}/monthly/2016010100/2021123100"
            )
            
            response = requests.get(api_url, verify=False, headers={'User-Agent': 'Mozilla/5.0'})
            data = response.json().get("items", [])
            
            for item in data:
                timestamp = item['timestamp']
                views = item['views']
                df_views.loc[df_views.page_title == url, timestamp] = views

        except Exception as e:
            print(f"Error en: {url} -> {e}")
            
    return df_views


In [ ]:
df = pd.read_csv('data/mental_disorders.tsv', sep='\t')
df

In [ ]:
df = df.dropna(subset=['Wikipedia'])
df = df.assign(Wikipedia=df['Wikipedia'].str.split(';')).explode('Wikipedia')
df = df.reset_index(drop=True)
df

In [ ]:
df_multi = get_multilingual_urls(df['Wikipedia'])
df_multi

In [ ]:
df_views = wikipedia_views(df_multi['multilingual_url'])
df_views

In [ ]:
df_annual = df_views.copy()
monthly_cols = [col for col in df_annual.columns if col != 'page_title']

yearly_groups = defaultdict(list)
for col in monthly_cols:
    year = str(col)[:4]
    yearly_groups[year].append(col)

for year, cols in yearly_groups.items():
    df_annual[year] = df_annual[cols].sum(axis=1)

df_annual = df_annual[['page_title'] + list(yearly_groups.keys())]
df_annual

In [ ]:
df_views['page_views'] = df_views.iloc[:, 1:].sum(axis=1)

In [ ]:
df_views

In [ ]:
df_multi_views = df_multi.merge(df_views, how='inner', left_on='multilingual_url', right_on='page_title')
df_multi_views

In [ ]:
df_multi_views = df_multi_views[['source_url', 'language', 'page_views']].groupby('source_url').agg({'page_views':sum, 'language':'nunique'})
df_multi_views.reset_index(inplace=True)
df_multi_views

In [ ]:
df_annual_multi_views = df_multi.merge(df_annual, how='inner', left_on='multilingual_url', right_on='page_title')
df_annual_multi_views

In [ ]:
df_annual_multi_views = df_annual_multi_views[['source_url', '2016', '2017', '2018', '2019', '2020', '2021']].groupby('source_url').sum()
df_annual_multi_views.reset_index(inplace=True)
df_annual_multi_views

In [ ]:
df = df.merge(df_multi_views, how='inner', left_on='Wikipedia', right_on='source_url')
df = df.merge(df_annual_multi_views, how='inner', left_on='Wikipedia', right_on='source_url')
df

In [ ]:
df.to_csv('data/mental_disorders_wikiviews.tsv', index=False, sep='\t')